# ADNI Data

---

### package imports and basic functions

---

In [2]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [3]:
from spectranorm import snm

In [4]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Extracting data

---

In [6]:
dataset = "ADNI"

In [4]:
data_info_df = pd.read_csv("/mnt/nas/CSC7/Yeolab/Data/ADNI/users_data/Sina/ADNI.csv")
data_info_df.shape


(16164, 7)

In [30]:
pd.notna(data_info_df['Scan_path']).sum()

9907

In [31]:
pd.notna(data_info_df['Scanner_info']).sum()

9925

In [36]:
num_unique = data_info_df[['RID']].drop_duplicates().shape[0]
print(num_unique)

2421


In [6]:
num_unique = data_info_df[['Site']].drop_duplicates().shape[0]
print(num_unique)

67


In [33]:
num_unique = data_info_df[['Scanner_info']].drop_duplicates().shape[0]
print(num_unique)

51


In [ ]:
adni_valid_subjects_dict = {}

for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = row["Scan_path"].split("/")[-1]
        adni_valid_subjects_dict[key] = {
            "unique_id": key,
            "participant_id": "_".join(key.split("_")[1:2]),
            "session_id": key.split("_")[-2],
            "site": str(row["Site"]),
            "sex": row["Sex"],
            "age": row["Age"],
            "scan_path": row["Scan_path"],
            "diagnosis": row["DX"],
        }

len(adni_valid_subjects_dict), list(adni_valid_subjects_dict.items())[:1]


In [40]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(adni_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]

    freesurfer_directory = adni_valid_subjects_dict[subject]["scan_path"]
    
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/ADNI/{sub_dir}/{subject}.thickness.fslr.npy"

    if not Path(thickness_fslr_output).exists():    
        # Compute fslr thickness
        transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
        np.save(
            ensure_dir(thickness_fslr_output),
            transformed_fslr_thickness.astype(np.float32)
        )


  0%|          | 0/9907 [00:00<?, ?it/s]

In [32]:
data_info_df[pd.notna(data_info_df['Scan_path'])][['DX']].value_counts(dropna=False)

DX 
MCI    3982
CN     3223
AD     1896
NaN     806
Name: count, dtype: int64

In [37]:
data_info_df[pd.notna(data_info_df['Scan_path'])][['Site']].value_counts(dropna=False)

Site
27      366
41      311
127     307
128     302
23      289
       ... 
301      20
341       6
121       5
177       4
132       4
Name: count, Length: 67, dtype: int64

In [ ]:
%%time
for idx, subject in enumerate(tqdm(adni_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    adni_valid_subjects_dict[subject]["subject_index"] = idx
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/ADNI/{sub_dir}/{subject}.thickness.fslr.npy"
    if Path(thickness_fslr_output).exists():
        adni_valid_subjects_dict[subject]["thickness"] = np.load(
            thickness_fslr_output,
        ).mean()
    else:
        adni_valid_subjects_dict[subject]["thickness"] = np.nan

len(adni_valid_subjects_dict), list(adni_valid_subjects_dict.items())[:1]


In [43]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Compute Euler Number
for idx, subject in enumerate(tqdm(adni_valid_subjects_dict)):
    if "euler_no" not in adni_valid_subjects_dict[subject]:
        sub_dir = f"{idx:02d}"[-2:]
        freesurfer_directory = adni_valid_subjects_dict[subject]["scan_path"]

        # Compute euler number
        adni_valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
            Path(freesurfer_directory)
        )


  0%|          | 0/9907 [00:00<?, ?it/s]

In [ ]:
for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = row["Scan_path"].split("/")[-1]
        adni_valid_subjects_dict[key]["participant_id"]= "_".join(key.split("_")[1:3])

len(adni_valid_subjects_dict), list(adni_valid_subjects_dict.items())[:1]


In [40]:
# Validity checks
for idx, subject in enumerate(tqdm(adni_valid_subjects_dict)):
    # if "validity_check" not in adni_valid_subjects_dict[subject]:
    adni_valid_subjects_dict[subject]["validity_check"] = (
        (adni_valid_subjects_dict[subject]["diagnosis"] in ['CN'])  # Exclude those with a diagnosis
        and
        not np.isnan(adni_valid_subjects_dict[subject]["age"])  # Exclude those missing age data
        and
        not np.isnan(adni_valid_subjects_dict[subject]["thickness"])  # Exclude those missing thickness data
        and
        not np.isnan(adni_valid_subjects_dict[subject]["euler_no"])  # Exclude those missing euler number
    )


  0%|          | 0/9907 [00:00<?, ?it/s]

In [41]:
import joblib

dataset = "ADNI"

joblib.dump(adni_valid_subjects_dict, ensure_dir(f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"))


['/home/sina/storage/Normative_Modeling/data/datasets/ADNI/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict = joblib.load(
    f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"
)

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [43]:
adni_valid_subjects_dict = valid_subjects_dict

In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[key]["age"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'thickness': [valid_subjects_dict[key]["thickness"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'sex': [valid_subjects_dict[key]["sex"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'site': [valid_subjects_dict[key]["site"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[key]["participant_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'euler_no': [valid_subjects_dict[key]["euler_no"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[key]["unique_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_index': [valid_subjects_dict[key]["subject_index"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [45]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(698, 9)

# ✅ Finished!
